In [127]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

def load_chroma_db():

    embeddings = OllamaEmbeddings(
        model="Qwen3-Embedding-0.6B-Q8_0:latest",
        temperature=0,
    )

    vector_store = Chroma(
        embedding_function = embeddings,
        collection_name = "chat_documents",
        persist_directory='.chroma_db',
    )
    return vector_store
vector_store = load_chroma_db()

In [128]:
collection = vector_store._collection
print(f"Number of documents: {collection.count()}")

Number of documents: 5105


In [129]:
documents = vector_store.similarity_search(
    "NonaSSA gc messages", 
    k=5, 
    filter={
        "$and": [
        {"start_date" : {"$lte" : filter_date}},
        {"other_person": "Damian"}
        ]})
first_document = documents[1] if documents else None
first_document.metadata

{'date_range': '2022-03-09 - 2022-03-10',
 'other_person': 'Damian',
 'source': 'whatsapp',
 'end_date': 20220310,
 'chat_name': 'Damian',
 'messages_count': 36,
 'chat_language': 'ro',
 'participants': "['Damian', 'Dan']",
 'part': 58,
 'is_groupchat': False,
 'start_date': 20220309}

### Instantiate Generation Model

In [142]:
from langchain_ollama import ChatOllama

chat_model = ChatOllama(
    model="qwen2.5:7b-instruct",
    num_predict=256,
    num_ctx=4096,          # Smaller context window
    system="You are a WhatsApp chat analysis agent that retrieves and analyzes conversation data to answer user questions about their chat history."
)

#### Setup RAG components

In [143]:
from typing_extensions import List, TypedDict, Annotated
from typing import Optional, Literal

class WhatsAppSearch(TypedDict):
    """WhatsApp chat search query with filtering."""
    #query: Annotated[str, ..., "Exacrtly"]
    participants: Annotated[
        Optional[List[str]], 
        ..., 
        "List of people participating in the conversation to filter by. Leave empty if not specified."
    ]
    time_period: Annotated[
        Optional[Literal["day", "month", "year"]],
        ...,
        "Type of time period: day, month, year, leave empty if no time period mentioned."
    ]
    date: Annotated[
        Optional[str],
        ...,
        "Time value in format: day -> dd/MM/yyyy, month -> MM/yyyy, year -> yyyy. Leave empty if date not mentioned."
    ]
    time_query: Annotated[
        Optional[Literal["within", "before", "after"]],
        ...,
        "Time query type: 'within' (during/in that specific period), 'before' (prior to), or 'after' (following). "
        "Use 'within' for queries like 'in 2023', 'during March', 'on that day'." \
        "Leave empty if not mentioned."
    ]
def analyze_query(question: str) -> WhatsAppSearch:
    """Analyze user question to extract search parameters."""
    structured_llm = chat_model.with_structured_output(schema=WhatsAppSearch)
    analysis_prompt = (
        "Extract from this WhatsApp question:\n"
        "- Person names mentioned\n" 
        "- Time references and whether it's before/after/within a time period\n"
        "- Convert dates to correct format in the date field: day=dd/MM/yyyy, month=MM/yyyy, year=yyyy\n"

        f"Question: {question}"
    )
    query_analysis = structured_llm.invoke(analysis_prompt)
    query_analysis['query'] = question
    return query_analysis

In [144]:
from datetime import datetime
import re


def parse_metadata_date_range(date_range_str: str):
    """Parse metadata date_range string like '2023-01-15 - 2023-01-20' into start and end dates."""
    try:
        start_str, end_str = date_range_str.split(' - ')
        start_date = datetime.strptime(start_str.strip(), '%Y-%m-%d').date()
        end_date = datetime.strptime(end_str.strip(), '%Y-%m-%d').date()
        return start_date, end_date
    except:
        return None, None


def extract_date_from_string(date_string):
    """Extract first valid date pattern from string and return yyyy/MM/dd format."""
    # Multiple date patterns to try
    patterns = [
        r'(\d{4})/(\d{1,2})/(\d{1,2})',  # yyyy/MM/dd or yyyy/M/d
        r'(\d{1,2})/(\d{1,2})/(\d{4})',  # dd/MM/yyyy or d/M/yyyy
        r'(\d{4})-(\d{1,2})-(\d{1,2})',  # yyyy-MM-dd
        r'(\d{1,2})-(\d{1,2})-(\d{4})',  # dd-MM-yyyy
        r'(\d{4})\.(\d{1,2})\.(\d{1,2})', # yyyy.MM.dd
        r'(\d{1,2})\.(\d{1,2})\.(\d{4})', # dd.MM.yyyy
    ]
    
    for pattern in patterns:
        match = re.search(pattern, str(date_string))
        if match:
            parts = match.groups()
            # Determine if it's yyyy/MM/dd or dd/MM/yyyy format
            if len(parts[0]) == 4:  # First part is year
                return f"{parts[0]}/{parts[1].zfill(2)}/{parts[2].zfill(2)}"
            else:  # First part is day
                return f"{parts[2]}/{parts[1].zfill(2)}/{parts[0].zfill(2)}"
    return None

def process_date_query(search_params):
    """Process date query parameters and return filtering criteria."""
    try:
        time_period = search_params.get('time_period')
        date_value = search_params.get('date')
        time_query = search_params.get('time_query')
        if not all([time_period, date_value, time_query]):
            return None
        
        # Extract robust date first
        extracted_date = extract_date_from_string(date_value)
        if extracted_date:
            # Convert yyyy/MM/dd to dd/MM/yyyy for your existing parser
            parts = extracted_date.split('/')
            date_value = f"{parts[2]}/{parts[1]}/{parts[0]}"
            search_params['date'] = date_value
        
        # Parse the date based on time_period
        if time_period == 'day':
            # Format: dd/MM/yyyy
            target_date = datetime.strptime(date_value, '%d/%m/%Y').date()
            return {
                'type': 'day',
                'date': target_date,
                'time_query': time_query
            }
        elif time_period == 'month':
            # Format: MM/yyyy
            try:
                month, year = date_value.split('/')
            except ValueError as e:
                print('Incorrect Format')
                target_date = datetime.strptime(date_value, '%d/%m/%Y').date()
                month = target_date.month
                year = target_date.year
            target_date = datetime(int(year), int(month), 1).date()
            return {
                'type': 'month',
                'date': target_date,
                'month': int(month),
                'year': int(year),
                'time_query': time_query
            }
        elif time_period == 'year':
            # Format: yyyy
            try:
                year = int(date_value)
            except ValueError as e:
                print(f'Incorrect Format')
                target_date = datetime.strptime(date_value,'%d/%m/%Y').date()
                year = target_date.year
            target_date = datetime(year, 1, 1).date()
            return {
                'type': 'year',
                'date': target_date,
                'year': year,
                'time_query': time_query
            }
    except ValueError as e:
        print(f' Could not load date, proceding with unfiltered chunks : {e}')
        return None
    

def date_filter_logic(doc_start_date, doc_end_date, date_criteria):
    """Apply date filtering logic based on criteria."""
    if not date_criteria:
        return True
    
    filter_type = date_criteria['type']
    time_query = date_criteria['time_query']
    
    if filter_type == 'day':
        target_date = date_criteria['date']
        
        if time_query == 'within':
            # Check if target date falls within document's date range
            return doc_start_date <= target_date <= doc_end_date
        elif time_query == 'before':
            # Document must end before target date
            return doc_end_date < target_date
        elif time_query == 'after':
            # Document must start after target date
            return doc_start_date > target_date
    
    elif filter_type == 'month':
        month = date_criteria['month']
        year = date_criteria['year']
        
        if time_query == 'within':
            # Check if any part of document overlaps with the target month
            doc_start_month = (doc_start_date.year, doc_start_date.month)
            doc_end_month = (doc_end_date.year, doc_end_date.month)
            target_month = (year, month)
            
            return doc_start_month <= target_month <= doc_end_month
        elif time_query == 'before':
            # Document must end before the target month
            return (doc_end_date.year, doc_end_date.month) < (year, month)
        elif time_query == 'after':
            # Document must start after the target month
            return (doc_start_date.year, doc_start_date.month) > (year, month)
    
    elif filter_type == 'year':
        year = date_criteria['year']
        
        if time_query == 'within':
            # Check if any part of document overlaps with the target year
            return doc_start_date.year <= year <= doc_end_date.year
        elif time_query == 'before':
            # Document must end before the target year
            return doc_end_date.year < year
        elif time_query == 'after':
            # Document must start after the target year
            return doc_start_date.year > year
    
    return True

In [ ]:
import datetime
date_criteria = {
    'type': 'year', 
    'date': datetime.date(2024, 1, 1), 
    'year': 2024, 
    'time_query': 'within'
}


metadata = {'date_range': '2022-03-09 - 2022-03-10',
 'other_person': 'Damian',
 'source': 'whatsapp',
 'end_date': 20220310,
 'chat_name': 'Damian',
 'messages_count': 36,
 'chat_language': 'ro',
 'participants': "['Damian', 'Dan']",
 'part': 58,
 'is_groupchat': False,
 'start_date': 20220309}

date_filter_logic(metadata['start_date'], metadata['end_date'], date_criteria)

AttributeError: 'int' object has no attribute 'year'

In [146]:
import ast
from langchain_core.tools import tool

def retrieve_date(date_criteria,search_params):
    
    # First retrieve all documents with similarity search
    all_docs = vector_store.similarity_search(
        search_params['query'],
        k=25,  # Get more docs to filter from
    )
    # Apply custom filtering
    filtered_docs = []
    for doc in all_docs:
        metadata = doc.metadata
        
        # Filter by participants if specified
        participant_match = True
        if search_params.get('participants'):
            doc_participants = metadata.get('participants', [])
            doc_participants = ast.literal_eval(doc_participants)

            participant_match = any(p.lower() in [dp.lower() for dp in doc_participants]
                                  for p in search_params['participants'])
            if not participant_match:
                continue

        # Get document date range
        doc_date_range = metadata.get('date_range')
        if not doc_date_range and date_criteria:
            continue
            
        date_match = True
        if date_criteria and doc_date_range:
            doc_start_date, doc_end_date = parse_metadata_date_range(doc_date_range)
            if not doc_start_date or not doc_end_date:
                continue
            
            # Apply date filtering logic
            date_match = date_filter_logic(doc_start_date, doc_end_date, date_criteria)
        
        if participant_match and date_match:
            filtered_docs.append(doc)

        if len(filtered_docs) >= 4:
            break
    
    return filtered_docs
    
@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """Retrieve information related to a query with intelligent filtering."""
    print(f"\n🔍 RETRIEVE DEBUG - Query: '{query}'")
    
    # Analyze the query for filtering parameters
    search_params = analyze_query(query)
    print(f"📊 Search params: {search_params}")
    
    # Process date query parameters
    
    date_criteria = process_date_query(search_params)
    print(f'Date Criteria {date_criteria}')
    if date_criteria:
        retrieved_docs = retrieve_date(date_criteria,search_params)
    else:
        # First retrieve all documents with similarity search
        retrieved_docs = vector_store.similarity_search(
        search_params['query'],
        k=5,
        )

    print(f'Number of chunks retrieved : {len(retrieved_docs)}')    
    serialized = "\n\n".join(
    f"Source: {doc.metadata.get('chat_name', 'Unknown')}\n"
    f"Date Range: {doc.metadata.get('date_range', 'Unknown')}\n"
    f"Content: {doc.page_content}"
    for doc in retrieved_docs
    )
    
    return serialized, retrieved_docs


In [147]:
from langchain_core.documents import Document
from typing_extensions import List,TypedDict
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage
from langgraph.graph import MessagesState, StateGraph

### Setting up orchestration of our Steps (retrieval and generation) using LangGraph
class State(TypedDict):
    question : str
    context : List[Document]
    answer: str

def query_or_respond(state: MessagesState):
    """ Generate tool call for retrieval or direct respond."""

    llm_with_tools = chat_model.bind_tools([retrieve])
    system_msg = SystemMessage(
        "You are a helpful assistant. When you don't know something, or are asked something about the user's chats or personal information, "
        "You MUST use the retrieve tool to find relevant information, which returns chunks of relevant WhatsApp Chat History. "
        "Use the retrieve tool for any question you cannot answer with high confidence."
        "When using the retrieve tool, make sure the query parameter is sufficiently detailed to refelct the original question, do not miss any details."
    )
    messages_with_system = [system_msg] + state["messages"]
    response = llm_with_tools.invoke(messages_with_system)
    print(f'State messages {response}')

    return {"messages": [response]}

### declare the tool, so that we can add it in a 'callable' node
tools = ToolNode([retrieve])

def generate(state: MessagesState):
    recent_tool_messages = []

    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]
    
    docs_content = "\n\n".join(doc.content for doc in tool_messages)
    
    system_message_content = (
    "You are an assistant for question-answering tasks about the user's past WhatsApp activities and interactions. "
    "Key points:\n"
    "- Use the retrieved chat excerpts to answer the question\n"
    "- Be specific about who said what and when, referencing chat metadata (names, dates, participants)\n"
    "- Focus on messages relevant to the question - multiple topics may appear in the same conversation\n"
    "- If context is unrelated or unclear, respond with 'I don't know about...'. Do NOT Make up facts\n"
    "- Keep answers simple, conversational, and don't directly refer to 'the conversations provided'\n"
    "- Use three sentences maximum\n"
    "- If context is not present, state that there is no information for that request."
    "- If there are no excerpts with 'Me' then assume I did not take part in the discussion."
    "\n\n"
    f"{docs_content}"
    )

    conversation = [
        message
        for message in state["messages"]
        if message.type in ("human","system")
        or (message.type =="ai" and not message.tool_calls)
    ]
    prompt = [SystemMessage(system_message_content)] + conversation   
    response = chat_model.invoke(prompt) 
    return {"messages" : [response]}


In [148]:
graph_builder = StateGraph(MessagesState)

In [149]:
from langgraph.graph import END
from langgraph.prebuilt import ToolNode, tools_condition
graph_builder = StateGraph(MessagesState)
graph_builder.add_node(query_or_respond)
graph_builder.add_node(tools)
graph_builder.add_node(generate)

graph_builder.set_entry_point("query_or_respond")
graph_builder.add_conditional_edges( 
    "query_or_respond",
    tools_condition,
    {END: END, "tools" : "tools"}
)
graph_builder.add_edge("tools","generate")
graph_builder.add_edge("generate",END)

graph = graph_builder.compile()

In [150]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
# specify thread_id
config = {"configurable" : {"thread_id" : "abc123"}}

In [151]:
#input_message = "What grades did I get on my university courses through the years?"
input_message = "Who did I talk to about going to the beach in 2024?"
for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
    config=config,
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Who did I talk to about going to the beach in 2024?
State messages content='' additional_kwargs={} response_metadata={'model': 'qwen2.5:7b-instruct', 'created_at': '2025-08-07T17:59:22.356427Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3302902417, 'load_duration': 707111750, 'prompt_eval_count': 243, 'prompt_eval_duration': 1283337666, 'eval_count': 28, 'eval_duration': 1308825875, 'model_name': 'qwen2.5:7b-instruct'} id='run--f9a6eb4a-5ae3-42cf-9cb6-61e9b5545514-0' tool_calls=[{'name': 'retrieve', 'args': {'query': 'beach 2024 conversation with contacts'}, 'id': '16c057bb-2006-40b6-ba6d-ae650c489052', 'type': 'tool_call'}] usage_metadata={'input_tokens': 243, 'output_tokens': 28, 'total_tokens': 271}
================================== Ai Message ==================================
Tool Calls:
  retrieve (16c057bb-2006-40b6-ba6d-ae650c489052)
 Call ID: 16c057bb-2006-40b6-ba6d-ae650c489052
  A

In [19]:
input_message = "What exactly did he say about Travis Scott?"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
    config=config,
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What exactly did he say about Travis Scott?
================================== Ai Message ==================================

Let's retrieve the specific conversation related to Travis Scott from Dan NL's chats.
Tool Calls:
  retrieve (62eb05f8-903e-4bb2-8833-eced676bd956)
 Call ID: 62eb05f8-903e-4bb2-8833-eced676bd956
  Args:
    query: Travis Scott tickets and discussions with Dan NL
================================= Tool Message =================================
Name: retrieve

Source: Dan NL
Content**Metadata of Conversation**
            Chat: Dan NL (part 33) | Date: 2024-05-09 - 2024-05-10 | Language: ro
            **End of Metadata of Conversation** 
 Dan NL: A tixketele la trav ies maine la 10
Dan NL: https://www.mojo.nl/concerten/travis-scott
Dan NL: Pahodu am gasit cat vor costa biletele
Dan NL: Da chestia e ca aici vor fi si locuri seated
Me: L-am intrebat si pe chino daxa vrea daca inca na r

#### Query Analysis - Tests


In [38]:
from typing_extensions import List, TypedDict, Annotated
from typing import Optional, Literal

class WhatsAppSearch(TypedDict):
    """WhatsApp chat search query with filtering."""
    query: Annotated[str, ..., "Search query to run."]
    participants: Annotated[
        Optional[List[str]], 
        ..., 
        "List of people participating in the conversation to filter by. Leave empty if not specified."
    ]
    time_period: Annotated[
        Optional[Literal["day", "month", "year"]],
        ...,
        "Type of time period: day, month, or year."
    ]
    date: Annotated[
        Optional[str],
        ...,
        "Time value in format: day -> dd/MM/yyyy, month -> MM/yyyy, year -> yyyy"
    ]
    time_query: Annotated[
        Optional[Literal["within", "before", "after"]],
        ...,
        "Time query type: 'within' (during/in that specific period), 'before' (prior to), or 'after' (following). "
        "Use 'within' for queries like 'in 2023', 'during March', 'on that day'."
    ]
def analyze_query(question: str) -> WhatsAppSearch:
    """Analyze user question to extract search parameters."""
    structured_llm = chat_model.with_structured_output(schema=WhatsAppSearch)
    analysis_prompt = (
        "Extract from this WhatsApp question:\n"
        "- Main search terms\n"
        "- Person names mentioned\n" 
        "- Time references and whether it's before/after/within a time period\n"
        "- Convert dates to correct format in the date field: day=dd/MM/yyyy, month=MM/yyyy, year=yyyy\n"

        f"Question: {question}"
    )
    query_analysis = structured_llm.invoke(analysis_prompt)
    return query_analysis

In [60]:
query = "Did I talk to Sim since April fools day in 2022?"
search_params = analyze_query(query)